# 02 · QLoRA fine-tuning and evaluation

Fine-tunes **Qwen3-4B-Instruct-2507** with QLoRA (`src/train.py`) on the 824 train emails, selects the checkpoint on the 105 val emails (val loss), then runs the adapter on val and test and scores it with `src/eval.py`. The test set is only read in section 5.

Runtime: **GPU T4**. Checkpoints and predictions go to Google Drive: if Colab disconnects, rerun the notebook from the top and training resumes from the last checkpoint.

⚠️ Until the hand review is frozen (`data/splits/test.jsonl`), test metrics are **provisional**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/Benarbiaa/email-extraction-qlora.git"
import os
if not os.path.exists("/content/email-extraction-qlora"):
    !git clone -q {REPO_URL} /content/email-extraction-qlora
%cd /content/email-extraction-qlora
!git pull -q && git log --oneline -1

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets jsonschema
import transformers, peft, trl, bitsandbytes, torch
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__,
      "| bnb", bitsandbytes.__version__, "| torch", torch.__version__)

In [ ]:
# Optional: Hugging Face token from Colab Secrets (key icon, name HF_TOKEN, notebook access on).
# Faster model downloads; needed later to push the adapter. Skipped if the secret is missing.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set from Colab secrets")
except Exception as e:
    print("no HF_TOKEN secret, downloading anonymously:", type(e).__name__)


## 2. Google Drive for checkpoints and predictions

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/email-extraction-qlora"
RUN = "qlora_v1"
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN}"
!mkdir -p {DRIVE_DIR}/preds {DRIVE_DIR}/tables {RUN_DIR}
!rm -rf results/preds && mkdir -p results && ln -s {DRIVE_DIR}/preds results/preds
GOLD = "data/splits/test.jsonl" if os.path.exists("data/splits/test.jsonl") else "data/splits/test_provisional.jsonl"
print("run dir:", RUN_DIR, "| test gold:", GOLD)

## 3. Smoke test (32 examples, 2 steps)
Checks token lengths, model loading, the LoRA setup, one eval and one checkpoint save in ~3 minutes, before the long run.

In [ ]:
!rm -rf /tmp/smoke
!python src/train.py --output-dir /tmp/smoke --run-name smoke --limit 32 --max-steps 2 --eval-steps 1 --save-steps 2
!rm -rf /tmp/smoke results/tables/train_smoke.json results/tables/train_log_smoke.csv

## 4. Train
2 epochs ≈ 104 optimizer steps (batch 2 × grad accum 8). Val loss every 10 steps; the best checkpoint is kept as `final`.
The first lines print the token-length check: training refuses to start if any example would be truncated.

In [ ]:
!python src/train.py --output-dir {RUN_DIR} --run-name {RUN}
!cp results/tables/train_{RUN}.json results/tables/train_log_{RUN}.csv {DRIVE_DIR}/tables/

In [ ]:
import pandas as pd
log = pd.read_csv(f"results/tables/train_log_{RUN}.csv")
ax = log.dropna(subset=["train_loss"]).plot(x="step", y="train_loss", alpha=0.5)
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o")
log.dropna(subset=["eval_loss"])[["step", "epoch", "eval_loss"]]

## 5. Evaluate the adapter: val first (sanity check), then test

In [ ]:
ADAPTER = f"{RUN_DIR}/final"
!python src/infer.py --mode finetuned --adapter {ADAPTER} --split data/splits/test_provisional.jsonl --limit 3 --out /tmp/ft_smoke.jsonl
!cat /tmp/ft_smoke.jsonl

In [ ]:
!python src/infer.py --mode finetuned --adapter {ADAPTER} --split data/splits/val.jsonl --out results/preds/ft_{RUN}_val.jsonl
!python src/eval.py --pred results/preds/ft_{RUN}_val.jsonl --gold data/splits/val.jsonl --name ft_{RUN}_val

In [ ]:
!python src/infer.py --mode finetuned --adapter {ADAPTER} --split {GOLD} --out results/preds/ft_{RUN}.jsonl
!python src/eval.py --pred results/preds/ft_{RUN}.jsonl --gold {GOLD} --name ft_{RUN}

## 6. Summary (copied to Drive)

In [ ]:
!cp results/tables/summary.csv results/tables/metrics_ft_*.json {DRIVE_DIR}/tables/
pd.read_csv("results/tables/summary.csv").set_index("name").T